<div class="lesson-banner">
<span class="lesson-kicker">Python course · 2-hour lesson</span>
<p>Read and write local data safely with `pathlib`, context managers, encodings, JSON, and CSV.</p>
</div>

## Learning objectives

- Build portable paths with `pathlib.Path`.
- Use context managers and explicit UTF-8 encoding.
- Choose JSON or CSV from the data shape.
- Validate external records before transformation.

::: {.callout-note}
### How to use this notebook
Read the explanation, predict each result, run the code, change the inputs, and complete the practice before revealing the solution.
:::


## Path operations and file lifecycle

`Path` joins locations without manual separators and exposes readable methods for existence, suffixes, iteration, and file I/O. A `with` block guarantees a file is closed even if processing fails. Treat filenames and encodings as explicit inputs.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as temporary:
    root = Path(temporary)
    notes_path = root / "notes" / "python.txt"
    notes_path.parent.mkdir(parents=True)
    notes_path.write_text("names\nfunctions\ncollections\n", encoding="utf-8")

    topics = notes_path.read_text(encoding="utf-8").splitlines()
    print(notes_path.name, notes_path.suffix, topics)


## JSON preserves nested structure

JSON supports objects, arrays, strings, numbers, booleans, and null. Python maps these to dictionaries, lists, strings, numbers, booleans, and `None`. Dates and custom classes need an explicit serialization policy. Use indentation for human-reviewed files and validate required keys after loading.


In [ ]:
import json

course = {
    "name": "Python",
    "hours": 45,
    "published": True,
    "modules": ["Core", "Professional", "Data and AI"],
}

payload = json.dumps(course, indent=2, ensure_ascii=False)
restored = json.loads(payload)
required = {"name", "hours", "modules"}
if missing := required - restored.keys():
    raise ValueError(f"missing keys: {sorted(missing)}")
print(payload)


## CSV represents rectangular records

CSV is widely interoperable but carries no reliable type information. Use `csv.DictReader` and `DictWriter`, open files with `newline=''`, and convert each field deliberately. A schema check should reject missing headers before row processing.


In [ ]:
import csv
import io

text = "name,score,active\nAsha,88,true\nRavi,73,false\n"
stream = io.StringIO(text)
reader = csv.DictReader(stream)

records = []
for row in reader:
    records.append({
        "name": row["name"],
        "score": float(row["score"]),
        "active": row["active"].casefold() == "true",
    })
print(records)


## Worked example: configurable course report

Keep configuration, input data, transformation, and output separate so each boundary is testable.


In [ ]:
import csv
import io
import json

config = json.loads('{"pass_mark": 60, "round_digits": 1}')
csv_text = "name,score\nAsha,88\nRavi,54\nMeera,91\n"

rows = csv.DictReader(io.StringIO(csv_text))
report = []
for row in rows:
    score = round(float(row["score"]), config["round_digits"])
    report.append({
        "name": row["name"],
        "score": score,
        "status": "pass" if score >= config["pass_mark"] else "review",
    })

print(json.dumps(report, indent=2))


## Practice lab

Complete these tasks without copying the solution. Test normal, boundary, and invalid inputs where relevant.

1. Write three course records to CSV and read them back with numeric hours.
2. Store application settings in JSON and verify three required keys.
3. List all `.ipynb` files directly inside a chosen directory.
4. Design a policy for malformed rows: stop, skip, or collect errors. Explain the trade-off.

::: {.callout-important}
### Practice standard
Your answer should be readable, deterministic, and divided into small functions when the task contains more than one rule.
:::


## Suggested solution

Open the folded code only after attempting every task.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import csv
import json

courses = [
    {"name": "Python", "hours": 45},
    {"name": "SQL", "hours": 30},
    {"name": "ML", "hours": 60},
]

with TemporaryDirectory() as temporary:
    root = Path(temporary)
    csv_path = root / "courses.csv"
    with csv_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["name", "hours"])
        writer.writeheader()
        writer.writerows(courses)

    with csv_path.open(encoding="utf-8", newline="") as handle:
        restored = [
            {"name": row["name"], "hours": int(row["hours"])}
            for row in csv.DictReader(handle)
        ]

    settings = {"input": str(csv_path), "strict": True, "encoding": "utf-8"}
    (root / "settings.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")
    print(restored)


## Knowledge check

**1. Why specify UTF-8?**

::: {.callout-note collapse="true"}
### Answer
It makes the encoding contract explicit and portable.
:::

**2. Why use `newline=''` with CSV?**

::: {.callout-note collapse="true"}
### Answer
It lets the CSV module handle newlines correctly.
:::

**3. Which format naturally supports nested data?**

::: {.callout-note collapse="true"}
### Answer
JSON.
:::


## Recap

- Use `Path`, not string concatenation.
- External text needs conversion and validation.
- Separate I/O from transformation.


<div class="lesson-nav">
<a href="05-text-and-regex.html"><i class="bi bi-arrow-left" aria-hidden="true"></i> Strings, Text Processing, and Regular Expressions</a>
<a href="07-errors-debugging-and-logging.html">Errors, Debugging, and Logging <i class="bi bi-arrow-right" aria-hidden="true"></i></a>
</div>
